# 현실의 데이터 04: 한국어 리뷰 감성 분석

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/datashop/04_korean_reviews.ipynb)

**분야**: 커머스 · 콘텐츠 &nbsp;|&nbsp; **이 분야에서 먼저 보는 문제**: 리뷰 분석 · 브랜드 모니터링 · 한국어 자연어처리

> 한국어 영화 리뷰 20만 개. 이 리뷰는 칭찬일까, 불평일까?

**데이터**: NSMC (Naver Sentiment Movie Corpus) (약 20MB). Lucy Park (2015), github.com/e9t/nsmc. 한국어 영화 리뷰 20만 개 (학습 15만 · 시험 5만), 긍정 · 부정 반반.

인턴 · 산학협력 과제에서 처음 받게 될 데이터와 닮은 공개 데이터다. 데이터를 열어 보고, 아래 예시처럼 문제 하나를 정해 기준선과 비교한 뒤, 맨 끝의 **나의 다섯 칸** 을 채운다 (1주 2차시 '문제 정형화와 평가지표').

| 칸 | 이 노트북의 예시 |
|---|---|
| ① 한 행 | 리뷰 한 문장 |
| ② 입력 X | 리뷰 글 |
| ③ 출력 y | 긍정 1 / 부정 0 |
| ④ 평가지표 | 정확도 (반반이라 쓸 만하다) |
| ⑤ 기준선 | 늘 '긍정' (50%) |

**논문 · 출처** (직접 열어 보자: 이 데이터를 누가, 왜, 어떻게 만들었나):
- [NSMC 저장소 (Lucy Park, 2015): 데이터 설명 · 만든 방법](https://github.com/e9t/nsmc)

In [1]:
import io, os, zipfile, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def download(url, path):
    """Download url to path once (skip if it already exists)"""
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    if not os.path.exists(path):
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (ML course data shop)"})
        with urllib.request.urlopen(req, timeout=600) as r, open(path, "wb") as f:
            f.write(r.read())
    return path

## 입장: 데이터 받기

In [2]:
BASE = "https://raw.githubusercontent.com/e9t/nsmc/master/"
train = pd.read_csv(download(BASE + "ratings_train.txt", "data/nsmc_train.txt"), sep="\t").dropna()
test = pd.read_csv(download(BASE + "ratings_test.txt", "data/nsmc_test.txt"), sep="\t").dropna()
print(train.shape, test.shape)
train.head()

(149995, 3) (49997, 3)


,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


## 열어 보기

긍정 비율, 리뷰 길이, 그리고 긍정 · 부정 예시.

In [3]:
print(f"positive share: train {train['label'].mean():.1%}, test {test['label'].mean():.1%}")
chars = train["document"].str.len()
print("characters per review: median", int(chars.median()), "max", chars.max())
for lab in [1, 0]:
    print("\nlabel", lab, train.loc[train["label"] == lab, "document"].sample(4, random_state=1).tolist())

positive share: train 49.9%, test 50.3%
characters per review: median 27 max 146

label 1 ['홍경민 연기 좋고 중견배우들은 말할것도 없이 굳!!', '눈을 사로잡는 독특한 소재, 어거지로 놀래키는 공포영화랑은 확연히 달랐다. 봉인된 유리문들이 하나하나 열릴때마다 느껴지는 섬뜩함과 짜릿함은 스릴러와 오락성 모두 채워줬다.', '재미있어요', '끝까지 즐겁고 행복하다']

label 0 ['아줌마들에게도 사랑은 필요하다?환타지가 필요하다면,해리-짐꾼을 읽어라.', '그냥 별로다....내시간 돌리도 ㅠㅠ', '구라씨 또 입으로 똥싸시네', '어설픈, 헛점 투성이의 스토리를 빛나는 연기로 포장한다. 아... 이정진씨 발연기는 제외']


## 문제 하나 골라 보기: 리뷰 → 긍정 / 부정

한국어는 띄어쓰기 단위로 자르면 '재밌다', '재밌어요', '재밌는' 이 다 다른 단어가 된다. 그래서 **글자 1~3개 묶음** 을 센다 (형태소 분석기 없이). 시간을 줄이려고 학습은 5만 개만.

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

small = train.sample(50000, random_state=0)
model = make_pipeline(TfidfVectorizer(analyzer="char_wb", ngram_range=(1, 3), min_df=3, max_features=200000),
                      LogisticRegression(max_iter=1000)).fit(small["document"], small["label"])
pred = model.predict(test["document"])
print(f"baseline (always positive): accuracy {test['label'].mean():.1%}")
print(f"char 1-3 grams + logistic : accuracy {np.mean(pred == test['label']):.1%}")
for s in ["진짜 재밌어요 강추", "시간 아깝다", "배우는 좋은데 스토리가 별로", "안 보면 후회함"]:
    print(f"  {s:20s} -> P(positive) = {model.predict_proba([s])[0, 1]:.2f}")

baseline (always positive): accuracy 50.3%
char 1-3 grams + logistic : accuracy 84.8%
  진짜 재밌어요 강추           -> P(positive) = 0.99
  시간 아깝다               -> P(positive) = 0.01
  배우는 좋은데 스토리가 별로      -> P(positive) = 0.12
  안 보면 후회함             -> P(positive) = 0.24


## 다른 문제로 바꿔 보기

- **브랜드 모니터링**: 리뷰 하나가 아니라 '하루 동안의 리뷰' 를 한 행으로 하면? y = 부정 리뷰 비율이 갑자기 늘었나 (경보).
- **별점**: 긍정 / 부정 두 칸 대신 별 1~10 개를 맞히면 회귀가 된다. 지표는?
- '배우는 좋은데 스토리가 별로' 처럼 섞인 리뷰는 무엇이 정답일까? 정답을 매긴 사람마다 다를 수 있다.

### 나의 다섯 칸

아래 표를 직접 채워 보자. 위의 예시와 다르게 정해도 된다.

| 칸 | 내가 정한 것 |
|---|---|
| ① 한 행은 무엇? | |
| ② 입력 X (예측할 때 알 수 있는 것만) | |
| ③ 출력 y | |
| ④ 평가지표 | |
| ⑤ 기준선 | |